# Cross-domain PLWCE α-sweep

CIFAR α*(IR) 법칙(α* = −1.017·ln(IR) + 8.071, R²=0.96)이 이미지 밖에서도 재현되는지 controlled 검증.
- **Cell 1**: setup + 함수 정의 (Drive 마운트, 로더 분리 import, MLP/train/LT 샘플러/sweep)
- **Cell 2**: tabular credit_card_fraud (binary)
- **Cell 3**: network CICIDS2018 (≥1000 필터 → 10-class)

train만 지수 LT 프로파일로 IR∈{10,20,50,100,200} 통제, test는 자연분포 고정. α*=test F1-macro argmax.

In [1]:
"""Cross-domain PLWCE alpha-sweep — does alpha*(IR) replicate outside CIFAR?

CIFAR-LT showed: optimal PLWCE alpha* DECREASES with imbalance ratio (IR),
fitting alpha* = -1.017*ln(IR) + 8.071 (R2=0.96) over IR in [10, 200].

This script replicates the *controlled* design in two non-image domains:
  - tabular  : credit_card_fraud (binary)
  - network  : CICIDS2018       (15-class)

Controlled design (same as CIFAR — only IR varies, everything else fixed):
  * Impose an exponential long-tail profile on the TRAIN set only:
        n_rank = N_HEAD * IR ** (-rank / (K - 1))      (rank 0 = head class)
    each class subsampled to min(target, available).
  * TEST set kept at its natural distribution (fixed across IRs) so F1-macro
    is comparable across IRs.
  * For each IR, sweep PLWCE alpha over a grid; alpha* = argmax mean test F1-macro.

Run on Colab:  !python alpha_sweep_crossdomain.py
Outputs: results/mlp/alpha_sweep_{domain}.json  +  printed alpha*(IR) table & fits.

NOTE on binary ccf: minority has only ~344 train samples, so at IR=200 the tail
is decimated to ~17 (noisy) — we average over SWEEP_SEEDS to reduce variance.
NOTE on CICIDS2018: tiny natural classes (4-17 samples) get clipped below their
exponential target, so we report the *realized* IR (head/min) alongside nominal.
"""
import os, sys, json, importlib.util
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

# ----------------------------------------------------------------------------- config
IR_GRID      = [10, 20, 50, 100, 200]
SWEEP_SEEDS  = [42, 43, 44]
SWEEP_EPOCHS = 50
BATCH_SIZE   = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3차 피드백 §4.6: "α와 ε 변화에 따른 Macro-F1, Worst-class Acc, Gradient Norm"
#   → PLWCE α 뿐 아니라 ES-LWCE ε sweep도 함께 돈다.
#   ε는 효과가 배수적이라 logspace (ε→0: LWCE 극한 / ε→∞: CE 극한).
ALPHA_GRID = [round(a, 3) for a in np.linspace(0.3, 6.0, 16)]        # PLWCE alpha
EPS_GRID   = [round(float(e), 4) for e in np.logspace(-1, 1, 16)]    # ES-LWCE eps
SWEEPS = [
    ('plwce',  'alpha', ALPHA_GRID),
    ('eslwce', 'eps',   EPS_GRID),
]

# --- paths (Colab 전용) ---
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/imbalanced-data-LWCE'
TAB_DATA = '/content/drive/MyDrive/imbalanced loss project/data'

RESULTS_DIR = f'{REPO}/network_data/results/mlp'   # 공용 결과 폴더
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Device: {device}  |  alpha grid: {ALPHA_GRID[0]}..{ALPHA_GRID[-1]} ({len(ALPHA_GRID)})')


# --------------------------------------------------------------- module loading helpers
def _load_module(name, path):
    """data_handler.py가 tabular_data/src·network_data/src 양쪽에 동명으로 존재 →
    sys.path 충돌 방지를 위해 명시 경로로 고유 이름으로 로드."""
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# get_clf_loss / GradLogger 는 REPO 루트의 공유 모듈
sys.path.insert(0, REPO)
from custom_losses import get_clf_loss
from experiment_utils import GradLogger, extended_metrics

tab_dh = _load_module('tab_data_handler', f'{REPO}/tabular_data/src/data_handler.py')
net_dh = _load_module('net_data_handler', f'{REPO}/network_data/src/data_handler.py')


# ------------------------------------------------------------------------- model + train
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers, in_dim = [], input_dim
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_eval(X_tr, y_tr, X_te, y_te, class_counts, num_classes,
               loss_name, params, seed):
    """손실 1개 학습 → test 지표 + gradient 통계 반환.
    3차 피드백 §4.6: F1뿐 아니라 Worst-class Acc / Gradient Norm도 필요.
      Worst-class는 per-class recall에서 사후 계산되지만 Gradient Norm은 학습 중에만 잡힌다."""
    torch.manual_seed(seed); np.random.seed(seed)
    tr = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
                    batch_size=min(BATCH_SIZE, len(X_tr)), shuffle=True, num_workers=0)
    model = MLP(X_tr.shape[1], num_classes).to(device)
    crit  = get_clf_loss(loss_name, class_counts, **params).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SWEEP_EPOCHS, eta_min=1e-5)
    glog  = GradLogger(class_counts, num_classes, device)

    for _ in range(SWEEP_EPOCHS):
        model.train()
        glog.reset()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            logits.retain_grad()          # 샘플별 로짓 기울기 관측 (Proposition 4)
            loss = crit(logits, yb)
            loss.backward()
            glog.update(logits, yb, model)   # opt.step() 전에 호출
            opt.step()
        sch.step()
    gstat = glog.epoch_end()              # 마지막 epoch 기준

    model.eval()
    with torch.no_grad():
        pred = []
        te = DataLoader(TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te)),
                        batch_size=4096, shuffle=False)
        for xb, _ in te:
            pred.append(model(xb.to(device)).argmax(1).cpu().numpy())
    y_pred = np.concatenate(pred)
    m = {'F1_Macro': float(f1_score(y_te, y_pred, average='macro', zero_division=0))}
    m.update(extended_metrics(y_te, y_pred, num_classes))   # G_Mean / Worst_Acc / Balanced_Acc
    m['grad_norm']  = gstat['grad_norm']
    m['grad_ratio'] = gstat['grad_ratio']
    del model, crit, opt
    import gc; gc.collect(); torch.cuda.empty_cache()
    return m


# --------------------------------------------------------------------- LT profile sampler
def make_lt_train(X, y, ir, n_head, seed=42):
    """지수 LT 프로파일: n_rank = n_head * ir^(-rank/(K-1)), rank0=다수.
    각 클래스를 min(target, 가용)으로 subsample. (realized_head, realized_min) 반환."""
    rng = np.random.RandomState(seed)
    classes, counts = np.unique(y, return_counts=True)
    order = classes[np.argsort(-counts)]            # head -> tail
    K = len(order)
    keep, realized = [], []
    for rank, cls in enumerate(order):
        target = n_head if K == 1 else max(1, int(round(n_head * ir ** (-rank / (K - 1)))))
        idx = np.where(y == cls)[0]
        if len(idx) > target:
            idx = rng.choice(idx, target, replace=False)
        keep.append(idx); realized.append(len(idx))
    keep = np.concatenate(keep); rng.shuffle(keep)
    return X[keep], y[keep], (max(realized), min(realized))


# ----------------------------------------------------------------------------- one sweep
def run_sweep(name, X_tr_full, y_tr_full, X_te, y_te, num_classes, n_head):
    print(f'\n{"="*64}\n[{name}]  K={num_classes}  N_HEAD={n_head}  '
          f'train_full={len(y_tr_full)}  test={len(y_te)}')
    rows, star = [], {}

    for loss_name, param, grid in SWEEPS:
        star[loss_name] = {}
        print(f'\n--- {loss_name} ({param}) ---')
        for ir in IR_GRID:
            f1_by_v, real_ir = {}, None
            for v in grid:
                agg = {'F1_Macro': [], 'Worst_Acc': [], 'G_Mean': [],
                       'grad_norm': [], 'grad_ratio': []}
                for sd in SWEEP_SEEDS:
                    Xs, ys, (rh, rm) = make_lt_train(X_tr_full, y_tr_full, ir, n_head, seed=sd)
                    cc = [int((ys == c).sum()) for c in range(num_classes)]
                    m = train_eval(Xs, ys, X_te, y_te, cc, num_classes,
                                   loss_name, {param: v}, sd)
                    for k in agg:
                        agg[k].append(m[k])
                    real_ir = rh / max(rm, 1)
                f1_by_v[v] = float(np.mean(agg['F1_Macro']))
                rows.append(dict(Domain=name, Loss=loss_name, IR=ir,
                                 realized_IR=round(real_ir, 1), param=param, value=float(v),
                                 **{k: float(np.mean(vs)) for k, vs in agg.items()}))
            best_v = max(f1_by_v, key=f1_by_v.get)
            star[loss_name][ir] = best_v
            print(f'  IR={ir:4d} (realized {real_ir:6.0f})  {param}*={best_v:.3f}  '
                  f'F1={f1_by_v[best_v]:.4f}')

        # fit x*(IR): raw-IR vs log-IR
        irs = np.array(IR_GRID, float)
        al  = np.array([star[loss_name][i] for i in IR_GRID])
        print(f'  {param}* by IR: {[round(star[loss_name][i], 3) for i in IR_GRID]}')
        for label, x in [('raw-IR ', irs), ('log-IR ', np.log(irs))]:
            if np.ptp(al) < 1e-9:
                print(f'  {label}: ({param}* 상수 — fit 생략)'); continue
            b1, b0 = np.polyfit(x, al, 1); yh = b1 * x + b0
            r2 = 1 - ((al - yh) ** 2).sum() / ((al - al.mean()) ** 2).sum()
            unit = 'IR' if 'raw' in label else 'ln(IR)'
            print(f'  {label}: {param}* = {b1:.4f}*{unit} + {b0:.4f}   R2={r2:.4f}')

    out = f'{RESULTS_DIR}/alpha_sweep_{name}.json'
    json.dump(rows, open(out, 'w'), indent=2)
    print(f'\n  saved: {out}  ({len(rows)} rows)')
    return star


# ----------------------------------------------------------------------------------- run
def sweep_tabular_ccf():
    X_tr, X_te, y_tr, y_te = tab_dh.load_dataset('credit_card_fraud', TAB_DATA,
                                                 test_size=0.3, random_state=42)
    X_tr = (X_tr.values if hasattr(X_tr, 'values') else X_tr).astype(np.float32)
    X_te = (X_te.values if hasattr(X_te, 'values') else X_te).astype(np.float32)
    y_tr = np.asarray(y_tr, dtype=np.int64); y_te = np.asarray(y_te, dtype=np.int64)
    # binary: 다수 고정 예산 N_HEAD=3440 → tail=3440/IR (IR=10→344[전량], IR=200→17)
    return run_sweep('credit_card_fraud', X_tr, y_tr, X_te, y_te,
                     num_classes=int(y_tr.max() + 1), n_head=3440)


def _filter_classes(X_tr, y_tr, X_te, y_te, min_count):
    """샘플 < min_count 인 클래스 제거 + 0..K'-1 relabel (train/test 동일).
    multiclass에서 4~17개짜리 극소 클래스가 realized IR을 고정시키는 문제 해결 —
    이 클래스들은 어떤 nominal IR에서도 subsample 불가라 IR 통제를 막음."""
    classes, counts = np.unique(y_tr, return_counts=True)
    keep = set(classes[counts >= min_count].tolist())
    remap = {c: i for i, c in enumerate(sorted(keep))}
    def f(X, y):
        m = np.isin(y, list(keep))
        return X[m], np.array([remap[v] for v in y[m]], dtype=np.int64)
    Xt, yt = f(X_tr, y_tr); Xe, ye = f(X_te, y_te)
    return Xt, yt, Xe, ye, len(keep)


def sweep_network_cicids2018():
    out = net_dh.load_network_dataset('CICIDS2018')
    X_tr, X_te, y_tr, y_te = out[0], out[1], out[2], out[3]
    X_tr = np.asarray(X_tr, dtype=np.float32); X_te = np.asarray(X_te, dtype=np.float32)
    y_tr = np.asarray(y_tr, dtype=np.int64);   y_te = np.asarray(y_te, dtype=np.int64)
    # 극소 클래스(4~17개) 제거 → 깔끔한 지수 프로파일로 IR 통제 가능 (min_count=1000 → ~10-class)
    X_tr, y_tr, X_te, y_te, K = _filter_classes(X_tr, y_tr, X_te, y_te, min_count=1000)
    print(f'[CICIDS2018] min_count=1000 필터 후 클래스 {K}개 '
          f'(min class train={min(int((y_tr==c).sum()) for c in range(K))})')
    # N_HEAD=8000, tail target=8000/IR (IR=200→40, 최소 클래스 1564개라 통제 가능)
    return run_sweep('CICIDS2018', X_tr, y_tr, X_te, y_te,
                     num_classes=K, n_head=8000)


Mounted at /content/drive
Device: cuda  |  alpha grid: 0.3..6.0 (16)


In [2]:
# Cell 2: tabular credit_card_fraud (binary) — α* vs IR
astar_ccf = sweep_tabular_ccf()


[credit_card_fraud]  K=2  N_HEAD=3440  train_full=199364  test=85443

--- plwce (alpha) ---
  IR=  10 (realized     10)  alpha*=0.300  F1=0.7426
  IR=  20 (realized     20)  alpha*=0.300  F1=0.8005
  IR=  50 (realized     50)  alpha*=0.300  F1=0.8461
  IR= 100 (realized    101)  alpha*=0.300  F1=0.8810
  IR= 200 (realized    202)  alpha*=0.300  F1=0.8871
  alpha* by IR: [np.float64(0.3), np.float64(0.3), np.float64(0.3), np.float64(0.3), np.float64(0.3)]
  raw-IR : (alpha* 상수 — fit 생략)
  log-IR : (alpha* 상수 — fit 생략)

--- eslwce (eps) ---
  IR=  10 (realized     10)  eps*=10.000  F1=0.7417
  IR=  20 (realized     20)  eps*=10.000  F1=0.7910
  IR=  50 (realized     50)  eps*=10.000  F1=0.8458
  IR= 100 (realized    101)  eps*=10.000  F1=0.8815
  IR= 200 (realized    202)  eps*=3.981  F1=0.8867
  eps* by IR: [10.0, 10.0, 10.0, 10.0, 3.981]
  raw-IR : eps* = -0.0309*IR + 11.1479   R2=0.7968
  log-IR : eps* = -1.5333*ln(IR) + 14.6576   R2=0.4699

  saved: /content/drive/MyDrive/imbalanced

In [3]:
# Cell 3: network CICIDS2018 (multiclass, ≥1000 필터) — α* vs IR
astar_cic = sweep_network_cicids2018()

100%|██████████| 1.60G/1.60G [01:36<00:00, 17.8MB/s]

Extracting files...


All files found (10):
  02-14-2018.csv  (358,223,333 bytes)
  02-15-2018.csv  (375,945,899 bytes)
  02-16-2018.csv  (333,723,605 bytes)
  02-20-2018.csv  (4,054,925,350 bytes)
  02-21-2018.csv  (328,893,673 bytes)
  02-22-2018.csv  (382,636,202 bytes)
  02-23-2018.csv  (382,840,456 bytes)
  02-28-2018.csv  (209,249,758 bytes)
  03-01-2018.csv  (107,842,858 bytes)
  03-02-2018.csv  (352,368,373 bytes)
Loading 10 CSV files (~60,000 rows each)...
Total rows loaded: 580,000
Dropping 5 mostly-inf columns

✓ Loaded: Train=166,261, Test=71,256, Input dim=78
Class distribution (train):
   0 Benign                             :  69,999 (42.10%)
   1 Bot                                :  10,927 (6.57%)
   2 Brute Force -Web                   :      17 (0.01%)
   3 Brute Force -XSS                   :      14 (0.01%)
   4 DDOS attack-HOIC                   :  27,223 (16.37%)
   5 DDOS attack-LOIC-UDP               :      73 (0.04%)
   6 DDoS attacks-LOIC-HTTP             :   2,950 (1.77%)
   7 Do

## 해석 가이드

CIFAR 참고: α* = −1.017·ln(IR) + 8.071 (R²=0.96, IR 10–200).

- **음의 log-IR 기울기 재현** → α*(IR) 법칙 일반성 입증 (이미지 전용 아님).
- 기울기 다르면 → 형태는 보편, 상수는 도메인 의존.
- binary ccf는 α* 절대값이 CIFAR와 다를 수 있음 → **단조 감소 트렌드 재현 여부**에 초점.
- ⚠️ log-IR 채택 근거는 R²(Δ작음, n=5)가 아니라 PLWCE log(n) 가중과의 이론 정합성.